In [53]:
import os
import glob
import pandas as pd
from functools import reduce

from chain import Chain

In [54]:
csv_folder = "cris_data/"
output_file = "collisions_db.csv"

In [55]:
# Function to clean column names
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [col.strip().replace(" ", "_") for col in df.columns]
    return df

collision_etl_chain = (Chain()
    .map(lambda file: pd.read_csv(file, skiprows=11)) # load files
    .pipe(lambda dfs: pd.concat(dfs, ignore_index=True)) # concatenate DataFrames
    .pipe(clean_column_names) # clean column names
)

def DataPreprocessor(csv_folder: str, function_chain: Chain):
    def fetch_csv_files(csv_folder):
        return glob.glob(os.path.join(csv_folder, "*.csv"))
    
    csv_files = fetch_csv_files(csv_folder)
    combined_df = function_chain(csv_files)
    return combined_df

combined_df = DataPreprocessor(csv_folder, collision_etl_chain)

In [56]:
combined_df.shape

(197544, 11)

In [57]:
combined_df.head()

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,On_System_Flag,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
0,18039083,1,C - POSSIBLE INJURY,2021,FRIDAY,03:00 - 03:59,Yes,22,H - HISPANIC,1 - MALE,1 - DRIVER
1,18039338,1,N - NOT INJURED,2021,FRIDAY,07:00 - 07:59,No,30,H - HISPANIC,2 - FEMALE,1 - DRIVER
2,18039360,1,N - NOT INJURED,2021,FRIDAY,11:00 - 11:59,No,55,W - WHITE,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
3,18039566,1,N - NOT INJURED,2021,FRIDAY,15:00 - 15:59,No,24,W - WHITE,2 - FEMALE,1 - DRIVER
4,18039566,1,N - NOT INJURED,2021,FRIDAY,15:00 - 15:59,No,73,W - WHITE,2 - FEMALE,1 - DRIVER


In [58]:
combined_df['Hour_of_Day'].dtype

<StringDtype(storage='python', na_value=nan)>

In [59]:
# --------------------------
# Extract first hour from range (e.g., "11:00 - 11:59")
# --------------------------
def extract_first_hour(hour_str: str) -> str:
    if pd.isna(hour_str):
        return pd.NA
    try:
        return str(hour_str).split(" - ")[0].strip()
    except:
        return pd.NA

# --------------------------
# Convert 24-hour string "HH:MM" to 12-hour integer
# --------------------------
def hour_24_to_12(hour_str: str) -> str:
    if pd.isna(hour_str):
        return pd.NA
    try:
        hour = int(hour_str.split(":")[0])
    except:
        return pd.NA

    if hour == 0:
        return "12 AM"
    elif hour < 12:
        return f"{hour} AM"
    elif hour == 12:
        return "12 PM"
    else:
        return f"{hour-12} PM"
    
def series_pipeline(*funcs):
    """
    Returns a function that applies a series of transformations to a Series.
    Each function should take a Series and return a Series.
    """
    def apply_pipeline(series: pd.Series) -> pd.Series:
        return reduce(lambda s, f: f(s), funcs, series)
    return apply_pipeline

# Create the pipeline: first extract first hour, then convert to 12-hour format
hour_pipeline = series_pipeline(
    lambda s: s.apply(extract_first_hour),
    lambda s: s.apply(hour_24_to_12)
)

# Apply the series pipeline and assign to the column
combined_df["Hour_of_Day"] = hour_pipeline(combined_df["Hour_of_Day"])
combined_df

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,On_System_Flag,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
0,18039083,1,C - POSSIBLE INJURY,2021,FRIDAY,3 AM,Yes,22,H - HISPANIC,1 - MALE,1 - DRIVER
1,18039338,1,N - NOT INJURED,2021,FRIDAY,7 AM,No,30,H - HISPANIC,2 - FEMALE,1 - DRIVER
2,18039360,1,N - NOT INJURED,2021,FRIDAY,11 AM,No,55,W - WHITE,1 - MALE,5 - DRIVER OF MOTORCYCLE TYPE VEHICLE
3,18039566,1,N - NOT INJURED,2021,FRIDAY,3 PM,No,24,W - WHITE,2 - FEMALE,1 - DRIVER
4,18039566,1,N - NOT INJURED,2021,FRIDAY,3 PM,No,73,W - WHITE,2 - FEMALE,1 - DRIVER
...,...,...,...,...,...,...,...,...,...,...,...
197539,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,41,H - HISPANIC,1 - MALE,1 - DRIVER
197540,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,33,H - HISPANIC,1 - MALE,2 - PASSENGER/OCCUPANT
197541,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,44,H - HISPANIC,2 - FEMALE,1 - DRIVER
197542,21292199,2,N - NOT INJURED,2026,MONDAY,12 PM,No,78,W - WHITE,1 - MALE,1 - DRIVER


In [60]:
def person_ethnicity_formatting(ethnicity_str: str) -> str:
    try:
        return ethnicity_str.split(" - ")[1].strip()
    except:
        return pd.NA

def person_gender_formatting(person_gender_str: str) -> str:
    try:
        return person_gender_str.split(" - ")[1].strip()
    except:
        return pd.NA

def person_type_formatting(person_type_str: str) -> str:
    try:
        return person_type_str.split(" - ")[1].strip()
    except:
        return pd.NA
    
# New mapping for Person_Type consolidation
PERSON_TYPE_MAP = {
    'DRIVER OF MOTORCYCLE TYPE VEHICLE': 'Motorcyclist',
    'PASSENGER/OCCUPANT ON MOTORCYCLE TYPE VEHICLE': 'Motorcyclist',
    'DRIVER': 'Motorist',
    'PASSENGER/OCCUPANT': 'Motorist',
    'PEDESTRIAN': 'Pedestrian',
    'PEDALCYCLIST': 'Cyclist',
    'OTHER (EXPLAIN IN NARRATIVE)': 'Other',
    'UNKNOWN': 'Unknown'
}

def person_type_consolidation(person_type_str: str) -> str:
    try:
        formatted = person_type_formatting(person_type_str)
        return PERSON_TYPE_MAP.get(formatted, 'Other')  # default to 'Other' if missing
    except:
        return pd.NA

pipeline_dict = {
    "Person_Ethnicity": series_pipeline(lambda s: s.apply(person_ethnicity_formatting)),
    "Person_Gender": series_pipeline(lambda s: s.apply(person_gender_formatting)),
    "Person_Type": series_pipeline(lambda s: s.apply(person_type_consolidation))
}

for column, pipeline in pipeline_dict.items():
    if column in combined_df.columns:
        combined_df[column] = pipeline(combined_df[column])

combined_df

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,On_System_Flag,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
0,18039083,1,C - POSSIBLE INJURY,2021,FRIDAY,3 AM,Yes,22,HISPANIC,MALE,Motorist
1,18039338,1,N - NOT INJURED,2021,FRIDAY,7 AM,No,30,HISPANIC,FEMALE,Motorist
2,18039360,1,N - NOT INJURED,2021,FRIDAY,11 AM,No,55,WHITE,MALE,Motorcyclist
3,18039566,1,N - NOT INJURED,2021,FRIDAY,3 PM,No,24,WHITE,FEMALE,Motorist
4,18039566,1,N - NOT INJURED,2021,FRIDAY,3 PM,No,73,WHITE,FEMALE,Motorist
...,...,...,...,...,...,...,...,...,...,...,...
197539,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,41,HISPANIC,MALE,Motorist
197540,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,33,HISPANIC,MALE,Motorist
197541,21292092,2,N - NOT INJURED,2026,MONDAY,8 AM,Yes,44,HISPANIC,FEMALE,Motorist
197542,21292199,2,N - NOT INJURED,2026,MONDAY,12 PM,No,78,WHITE,MALE,Motorist


In [61]:
filters_chain = (Chain()
    .pipe(lambda df: df.dropna(subset=["On_System_Flag"])) # drop rows where 'On System Flag' is NaN
    .pipe(lambda df: df.query("`On_System_Flag` == 'No'")) # filter rows where 'On System Flag' is 'No'
    .pipe(lambda df: df.query("`Crash_Severity` in ['A - SUSPECTED SERIOUS INJURY', 'K - FATAL INJURY']"))  # keep Fatal or Serious
)

combined_df = filters_chain(combined_df)
combined_df.drop(columns=["On_System_Flag"], inplace=True)
combined_df.to_csv("collisions_db.csv", index=False)
combined_df

,Crash_ID,Crash_Month,Crash_Severity,Crash_Year,Day_of_Week,Hour_of_Day,Person_Age,Person_Ethnicity,Person_Gender,Person_Type
341,18063119,1,A - SUSPECTED SERIOUS INJURY,2021,TUESDAY,11 AM,22,BLACK,MALE,Motorcyclist
684,18057364,1,K - FATAL INJURY,2021,TUESDAY,3 PM,26,HISPANIC,MALE,Motorcyclist
685,18057364,1,K - FATAL INJURY,2021,TUESDAY,3 PM,69,HISPANIC,FEMALE,Motorist
904,18062920,1,A - SUSPECTED SERIOUS INJURY,2021,THURSDAY,3 PM,26,HISPANIC,MALE,Motorcyclist
905,18062920,1,A - SUSPECTED SERIOUS INJURY,2021,THURSDAY,3 PM,38,WHITE,MALE,Motorist
...,...,...,...,...,...,...,...,...,...,...
196014,21285498,2,A - SUSPECTED SERIOUS INJURY,2026,FRIDAY,2 PM,60,HISPANIC,MALE,Motorist
196035,21265888,2,K - FATAL INJURY,2026,SATURDAY,5 PM,62,WHITE,FEMALE,Motorist
196036,21265888,2,K - FATAL INJURY,2026,SATURDAY,5 PM,57,HISPANIC,FEMALE,Pedestrian
196115,21266118,2,K - FATAL INJURY,2026,SUNDAY,2 AM,20,BLACK,FEMALE,Motorist


In [ ]:
combined_df.groupby(
        ["Crash_Severity", "Crash_Month", "Crash_Year"], as_index=False
    ).size().rename(columns={"size": "Crash_Count"}).to_csv("crash_counts_by_month.csv", index=False)

,Crash_Severity,Crash_Month,Crash_Year,Crash_Count
0,A - SUSPECTED SERIOUS INJURY,1,2021,12
1,A - SUSPECTED SERIOUS INJURY,1,2022,13
2,A - SUSPECTED SERIOUS INJURY,1,2024,15
3,A - SUSPECTED SERIOUS INJURY,1,2025,20
4,A - SUSPECTED SERIOUS INJURY,1,2026,24
...,...,...,...,...
82,K - FATAL INJURY,11,2024,1
83,K - FATAL INJURY,11,2025,2
84,K - FATAL INJURY,12,2021,3
85,K - FATAL INJURY,12,2022,1


In [ ]:
combined_df.groupby(
        ["Crash_Severity", "Person_Type", "Crash_Year"], as_index=False
    ).size().rename(columns={"size": "Crash_Count"}).to_csv("crash_counts_by_person_type.csv", index=False)

,Crash_Severity,Person_Type,Crash_Year,Crash_Count
0,A - SUSPECTED SERIOUS INJURY,Cyclist,2021,7
1,A - SUSPECTED SERIOUS INJURY,Cyclist,2022,1
2,A - SUSPECTED SERIOUS INJURY,Cyclist,2024,5
3,A - SUSPECTED SERIOUS INJURY,Cyclist,2025,4
4,A - SUSPECTED SERIOUS INJURY,Motorcyclist,2021,23
5,A - SUSPECTED SERIOUS INJURY,Motorcyclist,2022,26
6,A - SUSPECTED SERIOUS INJURY,Motorcyclist,2024,24
7,A - SUSPECTED SERIOUS INJURY,Motorcyclist,2025,24
8,A - SUSPECTED SERIOUS INJURY,Motorcyclist,2026,12
9,A - SUSPECTED SERIOUS INJURY,Motorist,2021,153


In [ ]:
combined_df.groupby(
        ["Crash_Severity", "Crash_Year", "Hour_of_Day", "Day_of_Week"], as_index=False
    ).size().rename(columns={"size": "Crash_Count"}).to_csv("crash_counts_by_time.csv", index=False)

,Crash_Severity,Crash_Year,Hour_of_Day,Day_of_Week,Crash_Count
0,A - SUSPECTED SERIOUS INJURY,2021,1 AM,FRIDAY,2
1,A - SUSPECTED SERIOUS INJURY,2021,1 AM,MONDAY,1
2,A - SUSPECTED SERIOUS INJURY,2021,1 AM,SATURDAY,2
3,A - SUSPECTED SERIOUS INJURY,2021,1 AM,SUNDAY,5
4,A - SUSPECTED SERIOUS INJURY,2021,1 PM,FRIDAY,4
...,...,...,...,...,...
351,K - FATAL INJURY,2026,1 AM,SUNDAY,4
352,K - FATAL INJURY,2026,2 AM,SUNDAY,4
353,K - FATAL INJURY,2026,2 AM,THURSDAY,6
354,K - FATAL INJURY,2026,4 PM,WEDNESDAY,4


In [ ]:
combined_df.groupby(
        ["Crash_Severity", "Crash_Year", "Person_Ethnicity"], as_index=False
    ).size().rename(columns={"size": "Crash_Count"})

,Crash_Severity,Crash_Year,Person_Ethnicity,Crash_Count
0,A - SUSPECTED SERIOUS INJURY,2021,BLACK,15
1,A - SUSPECTED SERIOUS INJURY,2021,HISPANIC,132
2,A - SUSPECTED SERIOUS INJURY,2021,UNKNOWN,6
3,A - SUSPECTED SERIOUS INJURY,2021,WHITE,54
4,A - SUSPECTED SERIOUS INJURY,2022,ASIAN,1
5,A - SUSPECTED SERIOUS INJURY,2022,BLACK,11
6,A - SUSPECTED SERIOUS INJURY,2022,HISPANIC,205
7,A - SUSPECTED SERIOUS INJURY,2022,OTHER,1
8,A - SUSPECTED SERIOUS INJURY,2022,UNKNOWN,8
9,A - SUSPECTED SERIOUS INJURY,2022,WHITE,59


In [76]:
combined_df.groupby(
        ["Crash_Severity", "Crash_Year", "Person_Age"], as_index=False
    ).size().rename(columns={"size": "Crash_Count"})

,Crash_Severity,Crash_Year,Person_Age,Crash_Count
0,A - SUSPECTED SERIOUS INJURY,2021,1,1
1,A - SUSPECTED SERIOUS INJURY,2021,10,2
2,A - SUSPECTED SERIOUS INJURY,2021,11,1
3,A - SUSPECTED SERIOUS INJURY,2021,15,5
4,A - SUSPECTED SERIOUS INJURY,2021,16,6
...,...,...,...,...
418,K - FATAL INJURY,2026,37,2
419,K - FATAL INJURY,2026,48,4
420,K - FATAL INJURY,2026,56,2
421,K - FATAL INJURY,2026,57,2


In [78]:
combined_df.groupby(
        ["Crash_Severity", "Crash_Year", "Person_Gender"], as_index=False
    ).size().rename(columns={"size": "Crash_Count"})

,Crash_Severity,Crash_Year,Person_Gender,Crash_Count
0,A - SUSPECTED SERIOUS INJURY,2021,FEMALE,76
1,A - SUSPECTED SERIOUS INJURY,2021,MALE,125
2,A - SUSPECTED SERIOUS INJURY,2021,UNKNOWN,6
3,A - SUSPECTED SERIOUS INJURY,2022,FEMALE,99
4,A - SUSPECTED SERIOUS INJURY,2022,MALE,183
5,A - SUSPECTED SERIOUS INJURY,2022,UNKNOWN,8
6,A - SUSPECTED SERIOUS INJURY,2024,FEMALE,77
7,A - SUSPECTED SERIOUS INJURY,2024,MALE,142
8,A - SUSPECTED SERIOUS INJURY,2024,UNKNOWN,8
9,A - SUSPECTED SERIOUS INJURY,2025,FEMALE,105
